# Lab 05 — Divergence Choice: One Variable, Measured Honestly

**Tier 2 lab.** Part A executes and asserts anywhere; Part B trains only when
`RUN_TRAINING = True`.

**The question.** Lab 01 §4 showed forward KL covering both modes of a cartoon bimodal teacher
and reverse KL seizing one. This lab asks what that picture is worth on a real student: train
the same pair, same data, same steps, same seed, with **only `beta` moving** — forward KL
(β=0), symmetric JSD (β=0.5), reverse KL (β=1) — and measure the things the theory says should
differ: entropy, diversity, confidence, agreement.

**The second, quieter subject is ablation discipline.** Most wrong conclusions in this field
come not from bad theory but from comparisons that varied two things, ran one seed, and
measured only loss. This lab's structure is the antidote, and it is three habits:

1. **One moving variable**, asserted mechanically, not eyeballed.
2. **Predictions registered before running.** Part A ends by writing a predictions dict to
   disk. Part C grades against it. You cannot fool yourself in hindsight about what you
   expected — the file has a timestamp.
3. **Two seeds minimum.** A difference smaller than the seed-to-seed spread is not a finding,
   it is noise wearing a costume.

The run matrix: 3 β values × 2 seeds = 6 short runs.

In [1]:
import sys, os, json, math
sys.path.insert(0, "../code")

import torch
import torch.nn.functional as F

from kd_core import (gjsd, kl_divergence, shift_for_next_token, top1_agreement,
                     mean_entropy, distinct_n, self_bleu, masked_mean)
from kd_pipeline import set_seed_everywhere, config_fingerprint, RunManifest

RUN_TRAINING = False        # <-- flip on the training box
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"torch {torch.__version__} | device: {device} | RUN_TRAINING: {RUN_TRAINING}")

BASE = dict(
    teacher="HuggingFaceTB/SmolLM2-360M-Instruct",
    student="HuggingFaceTB/SmolLM2-135M-Instruct",
    data="../data/lab03", seq_len=384, T=1.0,
    lr=3e-5, batch_size=8, grad_accum=4, max_steps=800, warmup_steps=40,
)
BETAS, SEEDS = (0.0, 0.5, 1.0), (17, 18)
MATRIX = [{**BASE, "beta": b, "seed": s} for b in BETAS for s in SEEDS]

def diff_keys(a, b):
    return {k for k in a.keys() | b.keys() if a.get(k) != b.get(k)}

for i, a in enumerate(MATRIX):
    for b in MATRIX[i+1:]:
        assert diff_keys(a, b) <= {"beta", "seed"}, "matrix must move beta and seed only"
print(f"{len(MATRIX)} runs, moving only beta x seed:",
      [(r['beta'], r['seed']) for r in MATRIX])

torch 2.13.0+cpu | device: cpu | RUN_TRAINING: False
6 runs, moving only beta x seed: [(0.0, 17), (0.0, 18), (0.5, 17), (0.5, 18), (1.0, 17), (1.0, 18)]


## Part A · 1 — What β actually interpolates, proven as a limit

Lab 01 verified the *direction* of TRL's β convention with four numbers. Here is the sharper
statement, worth proving once because it explains why small β behaves like a scaled forward KL
rather than like nothing: as β→0,

$$\mathrm{JSD}_\beta(p \,\|\, q) \;\approx\; \beta \cdot \mathrm{KL}(p \,\|\, q),$$

and symmetrically as β→1 with the reverse KL. The divergence itself vanishes at the endpoints —
it is the *ratio* JSD_β/β that converges. Practically: β=0.1 is not "10% of the way to reverse
KL" in any behavioral sense; it is still an almost-pure forward-KL objective at 1/10 scale, and
gradient-scale effects of changing β are easy to misread as behavioral ones. (TRL's trainers
handle β=0 and β=1 as exact special cases, so the endpoint rows in our matrix are exact, not
limits.)

In [2]:
g = torch.Generator().manual_seed(3)
zs = torch.randn(2, 6, 96, generator=g)
zt = torch.randn(2, 6, 96, generator=g)
m = torch.ones(2, 6, dtype=torch.bool)

fwd = float(kl_divergence(zs, zt, m, scale_by_T2=False))
rev = float(kl_divergence(zs, zt, m, direction="reverse", scale_by_T2=False))

for eps in (1e-2, 1e-3):
    lo = float(gjsd(zs, zt, m, beta=eps)) / eps
    hi = float(gjsd(zs, zt, m, beta=1 - eps)) / eps
    print(f"eps={eps:g}:  JSD_b/b -> {lo:.4f} (fwd KL {fwd:.4f})   "
          f"JSD_b/(1-b) -> {hi:.4f} (rev KL {rev:.4f})")
assert abs(float(gjsd(zs, zt, m, beta=1e-3)) / 1e-3 - fwd) / fwd < 0.02
assert abs(float(gjsd(zs, zt, m, beta=1 - 1e-3)) / 1e-3 - rev) / rev < 0.02
print("\nJSD_beta/beta -> forward KL and JSD_beta/(1-beta) -> reverse KL, verified")

eps=0.01:  JSD_b/b -> 1.0018 (fwd KL 1.0310)   JSD_b/(1-b) -> 0.9995 (rev KL 1.0293)
eps=0.001:  JSD_b/b -> 1.0279 (fwd KL 1.0310)   JSD_b/(1-b) -> 1.0260 (rev KL 1.0293)

JSD_beta/beta -> forward KL and JSD_beta/(1-beta) -> reverse KL, verified


## Part A · 2 — Trust the rulers before measuring with them

Part B judges runs on generation metrics. Each is verified here on inputs where the right
answer is computable by hand — the metric-audit habit: never report a number from a function
you have not fed a known case.

- `distinct_n`: unique n-grams / total n-grams. Two identical samples → low; all-distinct
  samples → 1.0.
- `self_bleu`: how much each sample's n-grams appear in the *others*. Identical samples → 1.0;
  disjoint samples → 0.0. Note the two metrics disagree about what "diverse" means at the
  margins, which is why the course logs both.
- `mean_entropy`: uniform over V → ln V exactly.

In [3]:
same = ["the cat sat on the mat", "the cat sat on the mat"]
diff = ["alpha beta gamma delta", "epsilon zeta eta theta"]

assert distinct_n(same, n=2) == 0.5, "two identical samples: every bigram appears twice"
assert distinct_n(diff, n=2) == 1.0, "fully distinct samples: all bigrams unique"
assert self_bleu(same, n=2) == 1.0, "identical samples are maximally self-similar"
assert self_bleu(diff, n=2) == 0.0, "disjoint samples share nothing"

V = 257
uniform = torch.zeros(1, 1, V)
H = mean_entropy(uniform, torch.ones(1, 1, dtype=torch.bool))
assert abs(H - math.log(V)) < 1e-5, "uniform entropy must equal ln V"
print(f"distinct_n / self_bleu endpoints verified; uniform entropy {H:.4f} == ln {V} "
      f"= {math.log(V):.4f}")

distinct_n / self_bleu endpoints verified; uniform entropy 5.5491 == ln 257 = 5.5491


## Part A · 3 — Register the predictions

Written now, graded in Part C, no edits in between. The theory being staked:

| metric (on generations) | forward KL (β=0) | JSD (β=0.5) | reverse KL (β=1) |
|---|---|---|---|
| mean entropy | **highest** | middle | **lowest** |
| distinct-3 | **highest** | middle | **lowest** |
| self-BLEU | **lowest** | middle | **highest** |
| top-1 agreement w/ teacher | lowest | middle | **highest** |

The mechanism, one line each: forward KL punishes the student for missing teacher mass, so the
student spreads (mode covering — high entropy, high diversity); reverse KL punishes mass where
the teacher has none, so the student concentrates on what it can match (mode seeking — confident,
repetitive, high agreement on the modes it kept); JSD is bounded on both sides and lands
between. The agreement row is the subtle one: reverse KL wins it *because* it abandoned the
tail, not despite it.

In [4]:
import time
PREDICTIONS = {
    "entropy":   {"order": ["0.0", "0.5", "1.0"], "direction": "decreasing in beta"},
    "distinct3": {"order": ["0.0", "0.5", "1.0"], "direction": "decreasing in beta"},
    "self_bleu": {"order": ["1.0", "0.5", "0.0"], "direction": "increasing in beta"},
    "agreement": {"order": ["1.0", "0.5", "0.0"], "direction": "increasing in beta"},
    "registered_unix_time": int(time.time()),
    "rule": "a beta effect counts only if it exceeds the max seed-to-seed spread",
}
os.makedirs("../runs/lab05", exist_ok=True)
with open("../runs/lab05/predictions.json", "w") as f:
    json.dump(PREDICTIONS, f, indent=2)
print("predictions registered and written to ../runs/lab05/predictions.json")
print("(Part C grades against this file; editing it after running is self-deception)")

predictions registered and written to ../runs/lab05/predictions.json
(Part C grades against this file; editing it after running is self-deception)


## Part B — Six runs and a sampling pass

One generic `train_one(beta, seed)` — Lab 03's loop with the loss swapped for `gjsd` — and a
`sample_and_measure` pass that generates from each trained student on held-out prompts at
temperature 1 (metrics about a model's *distribution* must be sampled at the temperature the
objective trained, not greedy-decoded, or entropy differences are hidden). The teacher scores
nothing here; it only provides logits during training and an agreement reference after.

In [5]:
from transformers import AutoModelForCausalLM, AutoTokenizer, get_cosine_schedule_with_warmup

def train_one(cfg):
    set_seed_everywhere(cfg["seed"])
    student = AutoModelForCausalLM.from_pretrained(cfg["student"], dtype=torch.bfloat16).to(device)
    teacher = AutoModelForCausalLM.from_pretrained(cfg["teacher"], dtype=torch.bfloat16).to(device).eval()
    for p in teacher.parameters():
        p.requires_grad_(False)
    tr = torch.load(os.path.join(cfg["data"], "train.pt"))
    batches = [{k: tr[k][i:i+cfg["batch_size"]].to(device)
                for k in ("input_ids", "mask")}
               for i in range(0, len(tr["input_ids"]), cfg["batch_size"])]
    opt = torch.optim.AdamW(student.parameters(), lr=cfg["lr"])
    sched = get_cosine_schedule_with_warmup(opt, cfg["warmup_steps"], cfg["max_steps"])
    step = 0
    while step < cfg["max_steps"]:
        for b in batches:
            s_logits = student(b["input_ids"]).logits
            with torch.no_grad():
                t_logits = teacher(b["input_ids"]).logits
            s_sh, t_sh, m_sh = shift_for_next_token(s_logits, t_logits, b["mask"])
            loss = gjsd(s_sh, t_sh, m_sh, beta=cfg["beta"], T=cfg["T"]) / cfg["grad_accum"]
            loss.backward()
            if (step + 1) % cfg["grad_accum"] == 0:
                torch.nn.utils.clip_grad_norm_(student.parameters(), 1.0)
                opt.step(); sched.step(); opt.zero_grad()
            step += 1
            if step >= cfg["max_steps"]:
                break
    return student, teacher

@torch.no_grad()
def sample_and_measure(student, teacher, cfg, n_prompts=32, max_new=96):
    tok = AutoTokenizer.from_pretrained(cfg["student"])
    ev = torch.load(os.path.join(cfg["data"], "eval.pt"))
    texts, entropies = [], []
    for i in range(n_prompts):
        plen = ev["prompt_lens"][i]
        prompt = ev["input_ids"][i:i+1, :plen].to(device)
        out = student.generate(prompt, do_sample=True, temperature=1.0, top_p=1.0,
                               max_new_tokens=max_new, pad_token_id=tok.eos_token_id)
        texts.append(tok.decode(out[0, plen:], skip_special_tokens=True))
    # distribution metrics on a held-out teacher-forced batch (comparable across arms)
    ids = ev["input_ids"][:64].to(device); m = ev["mask"][:64].to(device)
    s_logits = student(ids).logits; t_logits = teacher(ids).logits
    s_sh, t_sh, m_sh = shift_for_next_token(s_logits, t_logits, m)
    return {"entropy": mean_entropy(s_sh, m_sh),
            "distinct3": distinct_n(texts, 3),
            "self_bleu": self_bleu(texts, 3),
            "agreement": top1_agreement(s_sh, t_sh, m_sh)}

if RUN_TRAINING:
    results = []
    for cfg in MATRIX:
        student, teacher = train_one(cfg)
        metrics = sample_and_measure(student, teacher, cfg)
        row = {"beta": cfg["beta"], "seed": cfg["seed"], **metrics}
        results.append(row)
        print(row)
        del student, teacher
        if device == "cuda":
            torch.cuda.empty_cache()
    with open("../runs/lab05/results.json", "w") as f:
        json.dump(results, f, indent=2)
else:
    print("RUN_TRAINING=False — Part B compiled but did not execute.")
    print("6 runs x ~15-25 min on the training box; results land in ../runs/lab05/")

/usr/local/lib/python3.11/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


RUN_TRAINING=False — Part B compiled but did not execute.
6 runs x ~15-25 min on the training box; results land in ../runs/lab05/


## Part C — Grade against the registered predictions

Load `results.json`, average the two seeds per β, and for each metric compare the β ordering to
`predictions.json`. The grading rule was registered too: **a β effect is real only if the gap
between β arms exceeds the largest seed-to-seed spread within any arm.** Apply it mechanically.

**Expected ranges.** Entropy gaps between β=0 and β=1 of 0.1–0.5 nats on this pair; distinct-3
gaps of 0.02–0.10; agreement gaps of 1–4 points favoring reverse KL. The orderings should hold
for entropy and self-BLEU in nearly all healthy runs; the agreement ordering is the most
fragile (it depends on how much tail the β=1 student actually dropped) — a violation there
with clean entropy/diversity orderings is a *believable* result worth writing up, not a bug.

**Failure signatures.**

- *All six runs nearly identical on every metric.* Training was too short to differentiate the
  objectives, or the lr was too low — the arms never left the initialization's basin. Double
  `max_steps` before concluding "β doesn't matter."
- *β=1 entropy collapses toward zero and generations degenerate into loops.* That is not
  "reverse KL being reverse KL" — that is the entropy-collapse failure mode arriving early
  (Lab 07 treats it properly). Note the step at which it happened; it becomes your first
  data point for Lab 07's monitor thresholds.
- *Predicted orderings hold within seeds but not across them.* Your effect is smaller than
  your noise. The honest report is "indistinguishable at n=2"; the fix is more seeds, not a
  better story.

**Write the verdict** as three lines, one per prediction group: held / violated / within noise,
with the number that decided each. Then the SME question: given these measurements, which β
would you ship for a customer who wants a *reliable* assistant, and which for one who wants a
*creative* one — and what number from your own table justifies each answer?

## Exercises

1. **Add β=0.25 and β=0.75.** Is the entropy trend monotone in β, or does it step at the
   endpoints? (Recall A·1: intermediate β is not a behavioral interpolation.)
2. **Temperature interaction.** Rerun the β sweep at T=2. Softening the teacher gives the
   forward-KL student more tail to cover — which gaps widen?
3. **Where did the tail go?** For β=0 vs β=1 students, plot per-position student probability
   of the teacher's *second*-ranked token. Reverse KL's tail-dropping should be visible
   directly, not just through entropy.
4. **TVD as a control.** Add a `tvd`-trained arm. It is bounded like JSD but not an
   f-interpolation of the KLs — where does it land on your metric table, and does that match
   its geometry?